In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

from scipy.stats import randint, uniform, loguniform

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

from xgboost import XGBRegressor

In [ ]:
sold_path = Path("../data/post_outlier/CRMLSSold_cleaned_out.csv")
sold = pd.read_csv(sold_path)
sold

In [ ]:
sold.columns

### Defining Feature Set
Before defining the feature set, we must remove all possible leakage. We want to include the mortgage rates; however, it is joined by the CloseDate. Therefore, we must refactor the mortgage rate to only be based off of the listing date.

In [ ]:
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"


def add_listing_asof_rate(properties: pd.DataFrame) -> pd.DataFrame:
    properties = properties.copy()
    properties["_original_order"] = range(len(properties))

    properties["ListingContractDate"] = pd.to_datetime(
        properties["ListingContractDate"],
        errors="coerce",
    )

    mortgage = pd.read_csv(FRED_URL)
    mortgage = mortgage.rename(
        columns={
            "observation_date": "rate_observation_date",
            "MORTGAGE30US": "rate_30yr_fixed_asof_listing",
        }
    )

    mortgage["rate_observation_date"] = pd.to_datetime(
        mortgage["rate_observation_date"],
        errors="coerce",
    )
    mortgage["rate_30yr_fixed_asof_listing"] = pd.to_numeric(
        mortgage["rate_30yr_fixed_asof_listing"],
        errors="coerce",
    )

    mortgage = (
        mortgage
        .dropna(
            subset=[
                "rate_observation_date",
                "rate_30yr_fixed_asof_listing",
            ]
        )
        .sort_values("rate_observation_date")
    )

    valid = properties["ListingContractDate"].notna()

    matched = pd.merge_asof(
        properties.loc[valid].sort_values("ListingContractDate"),
        mortgage,
        left_on="ListingContractDate",
        right_on="rate_observation_date",
        direction="backward",
        # Strictly earlier observation avoids same-day availability ambiguity.
        allow_exact_matches=False,
        tolerance=pd.Timedelta(days=21),
    )

    missing_dates = properties.loc[~valid].copy()
    missing_dates["rate_observation_date"] = pd.NaT
    missing_dates["rate_30yr_fixed_asof_listing"] = pd.NA

    result = pd.concat([matched, missing_dates], ignore_index=True)
    result = result.sort_values("_original_order").drop(columns="_original_order")

    result["rate_age_days"] = (
        result["ListingContractDate"] - result["rate_observation_date"]
    ).dt.days

    return result

sold = add_listing_asof_rate(sold)

In [ ]:
sold.drop(sold[sold["ListingId"] == "SDC0001025SD"].index, inplace=True)
sold.drop(sold[sold["BathroomsTotalInteger"] > 45].index, inplace=True)

sold["ListingContractDate"] = pd.to_datetime(sold["ListingContractDate"])
sold["listing_year"] = sold["ListingContractDate"].dt.year
sold["listing_month"] = sold["ListingContractDate"].dt.month

In [ ]:
sold = sold[sold["BedroomsTotal"] >= 0]
sold = sold[sold["BathroomsTotalInteger"] >= 0]
sold = sold[sold["coordinates_in_california"] == True]
sold = sold[sold["ClosePrice"] > 0]

In [ ]:
quantitative_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "listing_month",
    "listing_year",
    "Latitude",
    "Longitude",
    "ParkingTotal",
    "rate_30yr_fixed_asof_listing"
]

qualitative_features = [
    "PropertySubType",
    "CountyOrParish",
]

features = quantitative_features + qualitative_features

In [ ]:
sold[features]

In [ ]:
model_df = sold[features + ["ClosePrice"]].copy()
model_df = model_df.dropna()

X = model_df[features]
y = np.log1p(model_df["ClosePrice"])

print("Rows used for modeling:", model_df.shape[0])

### Train/test split 80/20, transform categorical columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')), # in case missing data
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_transformer, qualitative_features)
    ],
    remainder="passthrough"
)

# Linear Regreesion

### Cross validation

In [ ]:
linear_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ('model', LinearRegression())])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(linear_pipeline, X_train, y_train, cv=kf, scoring="r2")
print(f"Cross-validation scores for Linear Regression: {cv_scores}")
print(f"Mean cross-validation score for Linear Regression: {cv_scores.mean():.3f}\n")

# Random Forest

### Cross validation

In [ ]:
rf_model = Pipeline(steps=[("preprocessor", preprocessor), ('model', RandomForestRegressor(n_estimators=100,
        max_depth=20,
        n_jobs=-1,
        random_state=42,
        verbose=1,))])

cv_scores = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="r2")
print(f"Cross-validation scores for Random Forest Regression: {cv_scores}")
print(f"Mean cross-validation score for Random Forest Regression: {cv_scores.mean():.3f}\n")

### Final evaluation

In [ ]:
rf_model.fit(X_train, y_train)
rf_predictions = np.expm1(rf_model.predict(X_test))
actual = np.expm1(y_test)

print(f"Random Forest Regression Performance on Test Set:")
print(f"R² Score: {r2_score(actual, rf_predictions):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(actual, rf_predictions)):.3f}")
print(f"MAE: {mean_absolute_error(actual, rf_predictions):.3f}")

# XGBoost

Final evaluation

In [ ]:
xgboost_model = Pipeline(steps=[("preprocessor", preprocessor), ('model', XGBRegressor(n_jobs=-1, tree_method="hist", random_state=42))])

param_grid = {
    "model__n_estimators": randint(200, 800),
    "model__max_depth": randint(6, 12),
    "model__learning_rate": loguniform(0.01, 0.2),
}

search = RandomizedSearchCV(
    xgboost_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="r2",
    cv=kf,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)
print("Best CV R²:", search.best_score_)
print("Best CV parameters:", search.best_params_)

In [ ]:
best_xgb = search.best_estimator_
best_xgb.fit(X_train, y_train)

xgb_predict = np.expm1(best_xgb.predict(X_test))
actual = np.expm1(y_test)

print(f"XGBoost Regression Performance on Test Set:")
print(f"R² Score: {r2_score(actual, xgb_predict):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(actual, xgb_predict)):.3f}")
print(f"MAE: {mean_absolute_error(actual, xgb_predict):.3f}")